# Inference And Gemini Judge

Notebook nay chi phuc vu `load data -> inference -> Gemini judge`.
Khong co buoc train.

In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("repo_root =", repo_root)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.utils.data_utils import load_project_dataset, load_saved_split_dataset
from src.utils.eval_utils import (
    evaluate_with_gemini,
    generate_responses,
    load_gemini_model_from_config,
    load_judge_prompts,
    save_records,
)
from src.utils.model_utils import load_config


In [ ]:
def summarize_model_device_map(model) -> str:
    hf_device_map = getattr(model, "hf_device_map", None)
    if hf_device_map:
        unique_targets = sorted({str(target) for target in hf_device_map.values()})
        return ", ".join(unique_targets)

    try:
        return str(next(model.parameters()).device)
    except StopIteration:
        return "unknown"


config = load_config(repo_root / "configs/eval_config.yaml")

# Chinh cac bien nay truoc khi chay.
MODEL_PATH = "models/sft_checkpoints/final"
SPLIT_NAME = "test_only"
NUM_SAMPLES = 10
MAX_NEW_TOKENS = 64
RESULTS_DIR = repo_root / "results/notebook_infer_judge"
RUN_JUDGE = True

config["evaluation"]["num_samples"] = NUM_SAMPLES
config["evaluation"]["max_new_tokens"] = MAX_NEW_TOKENS

print({
    "MODEL_PATH": MODEL_PATH,
    "SPLIT_NAME": SPLIT_NAME,
    "NUM_SAMPLES": NUM_SAMPLES,
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "RESULTS_DIR": str(RESULTS_DIR),
    "RUN_JUDGE": RUN_JUDGE,
})

In [ ]:
try:
    dataset = load_saved_split_dataset(SPLIT_NAME)
    dataset_source = f"saved split: {SPLIT_NAME}"
except Exception:
    dataset = load_project_dataset(
        config["evaluation"]["test_dataset"],
        split=config["evaluation"].get("source_split", "test"),
        prefer_local=True,
    )
    dataset_source = "fallback dataset"

print("dataset_source =", dataset_source)
print("dataset_size =", len(dataset))
dataset[0]

In [ ]:
resolved_model_path = repo_root / MODEL_PATH if not Path(MODEL_PATH).is_absolute() else Path(MODEL_PATH)

print("CUDA available =", torch.cuda.is_available())
print("CUDA device count =", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "all"))
print("torch CUDA version =", torch.version.cuda)

model = AutoModelForCausalLM.from_pretrained(
    str(resolved_model_path),
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(str(resolved_model_path), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("model_device_map =", summarize_model_device_map(model))

In [ ]:
responses = generate_responses(model, tokenizer, dataset, config)
infer_dir = RESULTS_DIR / "inference"
responses_json_path, responses_csv_path = save_records(responses, infer_dir, "generated_responses")

print("generated =", len(responses))
print("responses_json_path =", responses_json_path)
print("responses_csv_path =", responses_csv_path)
responses[0]

In [ ]:
evaluations = None
evaluation_json_path = None
evaluation_csv_path = None

if RUN_JUDGE:
    gemini_model = load_gemini_model_from_config(config)
    prompt_bundle = load_judge_prompts(config["metrics"]["prompt_file"])
    evaluations = evaluate_with_gemini(gemini_model, responses, prompt_bundle, config)
    judge_dir = RESULTS_DIR / "judge"
    evaluation_json_path, evaluation_csv_path = save_records(evaluations, judge_dir, "evaluation_results")
    print("evaluated =", len(evaluations))
    print("evaluation_json_path =", evaluation_json_path)
    print("evaluation_csv_path =", evaluation_csv_path)
else:
    print("RUN_JUDGE = False, chi chay inference.")

In [ ]:
if evaluations is not None:
    evaluations[0]
else:
    responses[0]